In [1]:
from copy import deepcopy
import json
import os

import numpy as np
import pandas as pd
from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import LabelEncoder



/usr/local/lib/python3.11/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:16: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


In [2]:
# Fix: make DATA_DIR robust to Kaggle's typical input mount points.
# (Score-neutral; prevents FileNotFoundError depending on dataset mount path.)
CANDIDATE_DATA_DIRS = [
    "../input/stanford-covid-vaccine",
    "/kaggle/input/stanford-covid-vaccine",
    "../input",
    "/kaggle/input",
]

DATA_DIR = None
for d in CANDIDATE_DATA_DIRS:
    if os.path.exists(os.path.join(d, "train.json")) and os.path.exists(
        os.path.join(d, "test.json")
    ):
        DATA_DIR = d
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate train.json/test.json in expected Kaggle input directories. "
        f"Tried: {CANDIDATE_DATA_DIRS}"
    )

train_path = os.path.join(DATA_DIR, "train.json")
test_path = os.path.join(DATA_DIR, "test.json")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

data_train = [json.loads(line) for line in open(train_path, "r")]
data_test = [json.loads(line) for line in open(test_path, "r")]
test_set = pd.read_csv(sample_path)

print("Using DATA_DIR:", DATA_DIR)
print(
    "train rows:",
    len(data_train),
    "test rows:",
    len(data_test),
    "sample rows:",
    len(test_set),
)



Using DATA_DIR: ../input/stanford-covid-vaccine
train rows: 2160 test rows: 240 sample rows: 25680


In [3]:
for jason in data_train:
    jason["step"] = list(range(jason["seq_scored"]))
    jason["sequence"] = list(jason["sequence"])
    jason["structure"] = list(jason["structure"])
    jason["predicted_loop_type"] = list(jason["predicted_loop_type"])



In [4]:
for jason in data_test:
    jason["step"] = list(range(jason["seq_scored"]))
    jason["sequence"] = list(jason["sequence"])
    jason["structure"] = list(jason["structure"])
    jason["predicted_loop_type"] = list(jason["predicted_loop_type"])



In [5]:
train = pd.json_normalize(
    data=data_train,
    record_path="reactivity",
    meta=["id", "signal_to_noise", "SN_filter", "seq_length", "seq_scored"],
)
train.rename(columns={0: "reactivity"}, inplace=True)

train["step"] = pd.json_normalize(data=data_train, record_path="step")
train["sequence"] = pd.json_normalize(data=data_train, record_path="sequence")
train["structure"] = pd.json_normalize(data=data_train, record_path="structure")
train["predicted_loop_type"] = pd.json_normalize(
    data=data_train, record_path="predicted_loop_type"
)

train["reactivity_error"] = pd.json_normalize(
    data=data_train, record_path="reactivity_error"
)

train["deg_Mg_pH10"] = pd.json_normalize(data=data_train, record_path="deg_Mg_pH10")
train["deg_error_Mg_pH10"] = pd.json_normalize(
    data=data_train, record_path="deg_error_Mg_pH10"
)

train["deg_pH10"] = pd.json_normalize(data=data_train, record_path="deg_pH10")
train["deg_error_pH10"] = pd.json_normalize(
    data=data_train, record_path="deg_error_pH10"
)

train["deg_Mg_50C"] = pd.json_normalize(data=data_train, record_path="deg_Mg_50C")
train["deg_error_Mg_50C"] = pd.json_normalize(
    data=data_train, record_path="deg_error_Mg_50C"
)

train["deg_50C"] = pd.json_normalize(data=data_train, record_path="deg_50C")
train["deg_error_50C"] = pd.json_normalize(data=data_train, record_path="deg_error_50C")

train.set_index(["id", "step"], inplace=True)



In [6]:
test = pd.json_normalize(
    data=data_test, record_path="sequence", meta=["id", "seq_length", "seq_scored"]
)
test.rename(columns={0: "sequence"}, inplace=True)

test["step"] = pd.json_normalize(data=data_test, record_path="step")
test["sequence"] = pd.json_normalize(data=data_test, record_path="sequence")
test["structure"] = pd.json_normalize(data=data_test, record_path="structure")
test["predicted_loop_type"] = pd.json_normalize(
    data=data_test, record_path="predicted_loop_type"
)

test.set_index(["id", "step"], inplace=True)



In [7]:
train



reactivity signal_to_noise SN_filter seq_length seq_scored  \
id           step                                                               
id_001f94081 0         0.3297           6.894         1        107         68   
             1         1.5693           6.894         1        107         68   
             2         1.1227           6.894         1        107         68   
             3         0.8686           6.894         1        107         68   
             4         0.7217           6.894         1        107         68   
...                       ...             ...       ...        ...        ...   
id_fff546103 63        0.0272           6.545         1        107         68   
             64        0.0382           6.545         1        107         68   
             65        0.0381           6.545         1        107         68   
             66       -0.0066           6.545         1        107         68   
             67        0.0706           6.545         1        107         68   

                  sequence structure predicted_loop_type  reactivity_error  \
id           step                                                            
id_001f94081 0           G         .                   E            0.1359   
             1           G         .                   E            0.2070   
             2           A         .                   E            0.1633   
             3           A         .                   E            0.1452   
             4           A         .                   E            0.1314   
...                    ...       ...                 ...               ...   
id_fff546103 63          A         (                   S            0.0267   
             64          A         (                   S            0.0219   
             65          G         (                   S            0.0218   
             66          G         (                   S            0.0220   
             67          U         .                   H            0.0287   

                   deg_Mg_pH10  deg_error_Mg_pH10  deg_pH10  deg_error_pH10  \
id           step                                                             
id_001f94081 0          0.7556             0.2613    2.3375          0.2631   
             1          2.9830             0.3842    3.5060          0.2860   
             2          0.2526             0.1372    0.3008          0.0964   
             3          1.3789             0.2531    1.0108          0.1574   
             4          0.6376             0.1798    0.2635          0.1000   
...                        ...                ...       ...             ...   
id_fff546103 63         0.5059             0.0626    0.4998          0.0996   
             64         0.2025             0.0411    0.3057          0.0760   
             65         0.1827             0.0391    0.1642          0.0611   
             66         0.3261             0.0509    0.1654          0.0719   
             67         0.4529             0.0565    0.4840          0.0936   

                   deg_Mg_50C  deg_error_Mg_50C  deg_50C  deg_error_50C  
id           step                                                        
id_001f94081 0         0.3581            0.1501   0.6382         0.2167  
             1         2.9683            0.2750   3.4773         0.3475  
             2         0.2589            0.0947   0.9988         0.1880  
             3         1.4552            0.1866   1.3228         0.2124  
             4         0.7244            0.1369   0.7877         0.1703  
...                       ...               ...      ...            ...  
id_fff546103 63        0.2870            0.0638   0.1820         0.0809  
             64        0.2022            0.0520   0.2038         0.0748  
             65        0.2047            0.0520   0.1120         0.0612  
             66        0.2447            0.0595   0.1186         0.0723  
             67        0.1802     

In [8]:
test



sequence seq_length seq_scored structure predicted_loop_type
id           step                                                             
id_00b436dec 0.0         G        107         68         .                   E
             1.0         G        107         68         .                   E
             2.0         A        107         68         .                   E
             3.0         A        107         68         .                   E
             4.0         A        107         68         .                   E
...                    ...        ...        ...       ...                 ...
id_ff103a85c NaN         A        107         68         .                   E
             NaN         C        107         68         .                   E
             NaN         A        107         68         .                   E
             NaN         A        107         68         .                   E
             NaN         C        107         68         .                   E

[25680 rows x 5 columns]

In [9]:
np.random.seed(2020)



In [10]:
TARGET_COLS = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]

X_train_enc = deepcopy(train)
X_test_enc = deepcopy(test)

category_cols_train = [
    c
    for c in X_train_enc.columns
    if (X_train_enc[c].dtype == "object" and c not in TARGET_COLS)
]
category_cols_test = [
    c
    for c in X_test_enc.columns
    if (X_test_enc[c].dtype == "object" and c not in TARGET_COLS)
]
cols_to_encode = sorted(set(category_cols_train).intersection(category_cols_test))

print("Train object cols:", category_cols_train)
print("Test object cols:", category_cols_test)
print("Encoding cols:", cols_to_encode)

encoders = {}
for col in cols_to_encode:
    le = LabelEncoder()
    combined = pd.concat(
        [X_train_enc[col].astype(str), X_test_enc[col].astype(str)],
        axis=0,
        ignore_index=True,
    )
    le.fit(combined)
    X_train_enc[col] = le.transform(X_train_enc[col].astype(str)).astype(np.int16)
    X_test_enc[col] = le.transform(X_test_enc[col].astype(str)).astype(np.int16)
    encoders[col] = le



Train object cols: ['signal_to_noise', 'SN_filter', 'seq_length', 'seq_scored', 'sequence', 'structure', 'predicted_loop_type']
Test object cols: ['sequence', 'seq_length', 'seq_scored', 'structure', 'predicted_loop_type']
Encoding cols: ['predicted_loop_type', 'seq_length', 'seq_scored', 'sequence', 'structure']


In [11]:
X = X_train_enc.drop(TARGET_COLS, axis=1)
y = X_train_enc.loc[:, TARGET_COLS]

missing_in_test = [c for c in X.columns if c not in X_test_enc.columns]
for c in missing_in_test:
    X_test_enc[c] = 0.0

X_test_enc = X_test_enc[X.columns]

X = X.apply(pd.to_numeric, errors="coerce").astype(np.float32)
X_test_enc = X_test_enc.apply(pd.to_numeric, errors="coerce").astype(np.float32)

X = X.fillna(0.0)
X_test_enc = X_test_enc.fillna(0.0)

y = y.apply(pd.to_numeric, errors="coerce").astype(np.float32).fillna(0.0)

print("X shape:", X.shape, "X_test shape:", X_test_enc.shape, "y shape:", y.shape)



X shape: (146880, 12) X_test shape: (25680, 12) y shape: (146880, 5)


In [12]:
hgbr = MultiOutputRegressor(
    HistGradientBoostingRegressor(
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=10,
        learning_rate=0.0025,
        tol=1e-5,
        verbose=0,
    ),
    n_jobs=4,
)



In [13]:
unique_ids = X.index.get_level_values("id").unique()
train_ids, valid_ids = train_test_split(unique_ids, test_size=0.2, random_state=2020)

X_train = X.loc[(train_ids, slice(None)), :]
y_train = y.loc[(train_ids, slice(None)), :]

X_valid = X.loc[(valid_ids, slice(None)), :]
y_valid = y.loc[(valid_ids, slice(None)), :]



In [14]:
X_train



signal_to_noise  SN_filter  seq_length  seq_scored  \
id           step                                                       
id_d780588bd 0               3.236        1.0         0.0         0.0   
             1               3.236        1.0         0.0         0.0   
             2               3.236        1.0         0.0         0.0   
             3               3.236        1.0         0.0         0.0   
             4               3.236        1.0         0.0         0.0   
...                            ...        ...         ...         ...   
id_647f39862 63              6.050        1.0         0.0         0.0   
             64              6.050        1.0         0.0         0.0   
             65              6.050        1.0         0.0         0.0   
             66              6.050        1.0         0.0         0.0   
             67              6.050        1.0         0.0         0.0   

                   sequence  structure  predicted_loop_type  reactivity_error  \
id           step                                                               
id_d780588bd 0          3.0        0.0                  5.0            0.3798   
             1          1.0        0.0                  5.0            0.4650   
             2          1.0        0.0                  5.0            0.3787   
             3          2.0        2.0                  2.0            0.1668   
             4          0.0        2.0                  2.0            0.3581   
...                     ...        ...                  ...               ...   
id_647f39862 63         3.0        0.0                  5.0            0.0303   
             64         2.0        0.0                  5.0            0.0339   
             65         3.0        0.0                  5.0            0.0248   
             66         3.0        2.0                  2.0            0.0369   
             67         3.0        2.0                  2.0            0.0392   

                   deg_error_Mg_pH10  deg_error_pH10  deg_error_Mg_50C  \
id           step                                                        
id_d780588bd 0                0.5689          0.5216            0.2791   
             1                1.4747          0.5042            0.4349   
             2                0.9515          0.3977            0.3897   
             3                0.4762          0.2320            0.1835   
             4                0.9216          0.2954            0.2862   
...                              ...             ...               ...   
id_647f39862 63               0.0403          0.0665            0.0358   
             64               0.0492          0.0781            0.0489   
             65               0.0235          0.0533            0.0306   
             66               0.0433          0.0778            0.0503   
             67               0.0332          0.0619            0.0400   

                   deg_error_50C  
id           step                 
id_d780588bd 0            0.3447  
             1            0.4307  
             2            0.4805  
             3            0.3823  
             4            0.3607  
...                          ...  
id_647f39862 63           0.0654  
             64           0.0665  
             65           0.0502  
             66           0.0803  
             67           0.0769  

[117504 rows x 12 columns]

In [15]:
hgbr.fit(X_train, y_train)



MultiOutputRegressor(estimator=HistGradientBoostingRegressor(early_stopping=True,
                                                             learning_rate=0.0025,
                                                             max_iter=1000,
                                                             tol=1e-05),
                     n_jobs=4)

In [16]:
y_pred = hgbr.predict(X_valid)



In [17]:
print(mean_squared_error(y_valid, y_pred, squared=False))



0.855341923184568


In [18]:
X_train_enc[X_test_enc.columns]



signal_to_noise SN_filter  seq_length  seq_scored  sequence  \
id           step                                                               
id_001f94081 0              6.894         1           0           0         2   
             1              6.894         1           0           0         2   
             2              6.894         1           0           0         0   
             3              6.894         1           0           0         0   
             4              6.894         1           0           0         0   
...                           ...       ...         ...         ...       ...   
id_fff546103 63             6.545         1           0           0         0   
             64             6.545         1           0           0         0   
             65             6.545         1           0           0         2   
             66             6.545         1           0           0         2   
             67             6.545         1           0           0         3   

                   structure  predicted_loop_type  reactivity_error  \
id           step                                                     
id_001f94081 0             2                    1            0.1359   
             1             2                    1            0.2070   
             2             2                    1            0.1633   
             3             2                    1            0.1452   
             4             2                    1            0.1314   
...                      ...                  ...               ...   
id_fff546103 63            0                    5            0.0267   
             64            0                    5            0.0219   
             65            0                    5            0.0218   
             66            0                    5            0.0220   
             67            2                    2            0.0287   

                   deg_error_Mg_pH10  deg_error_pH10  deg_error_Mg_50C  \
id           step                                                        
id_001f94081 0                0.2613          0.2631            0.1501   
             1                0.3842          0.2860            0.2750   
             2                0.1372          0.0964            0.0947   
             3                0.2531          0.1574            0.1866   
             4                0.1798          0.1000            0.1369   
...                              ...             ...               ...   
id_fff546103 63               0.0626          0.0996            0.0638   
             64               0.0411          0.0760            0.0520   
             65               0.0391          0.0611            0.0520   
             66               0.0509          0.0719            0.0595   
             67               0.0565          0.0936            0.0515   

                   deg_error_50C  
id           step                 
id_001f94081 0            0.2167  
             1            0.3475  
             2            0.1880  
             3            0.2124  
             4            0.1703  
...                          ...  
id_fff546103 63           0.0809  
             64           0.0748  
             65           0.0612  
             66           0.0723  
             67           0.0892  

[146880 rows x 12 columns]

In [19]:
BLEND_OOF_WEIGHT = (
    0.35  # 0 => only full-fit (likely better); 1 => only OOF-fit (likely worse)
)

hgbr_full = MultiOutputRegressor(
    HistGradientBoostingRegressor(
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=10,
        learning_rate=0.0025,
        tol=1e-5,
        verbose=0,
    ),
    n_jobs=4,
)
hgbr_full.fit(X, y)
y_pred_full = hgbr_full.predict(X_test_enc)

hgbr_oof = MultiOutputRegressor(
    HistGradientBoostingRegressor(
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=10,
        learning_rate=0.0025,
        tol=1e-5,
        verbose=0,
    ),
    n_jobs=4,
)
hgbr_oof.fit(X_train, y_train)
y_pred_oof = hgbr_oof.predict(X_test_enc)

y_pred_2 = (1.0 - BLEND_OOF_WEIGHT) * y_pred_full + BLEND_OOF_WEIGHT * y_pred_oof



In [20]:
y_pred_2.shape



(25680, 5)

In [21]:
target_cols = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]

pred_df_68 = pd.DataFrame(
    y_pred_2, index=X_test_enc.index, columns=target_cols
).reset_index()

test_meta = pd.DataFrame(
    [
        {
            "id": d["id"],
            "seq_length": int(d["seq_length"]),
            "seq_scored": int(d["seq_scored"]),
        }
        for d in data_test
    ]
)

pred_df_68 = pred_df_68.merge(test_meta, on="id", how="left")

rows = []
# Fix: use only the first seq_scored predictions (68) before padding to seq_length (107).
# Previously, when duplicates/misalignment happened, grp could exceed seq_scored, causing
# length mismatches during id_seqpos construction.
for _id, grp in pred_df_68.groupby("id", sort=False):
    grp = grp.sort_values("step").reset_index(drop=True)
    seq_len = int(grp["seq_length"].iloc[0])
    seq_scored = int(grp["seq_scored"].iloc[0])

    base = grp[target_cols].to_numpy(dtype=np.float32)

    # Keep exactly seq_scored rows (what the model predicts by construction).
    if base.shape[0] >= seq_scored:
        base = base[:seq_scored, :]
    else:
        # Shouldn't happen, but guard: pad within scored length if needed.
        pad_scored = np.repeat(
            base[[-1], :], repeats=(seq_scored - base.shape[0]), axis=0
        )
        base = np.vstack([base, pad_scored])

    # Now extend to seq_length for submission requirement (positions beyond seq_scored are unscored).
    if seq_len > seq_scored:
        pad = np.repeat(base[[-1], :], repeats=(seq_len - seq_scored), axis=0)
        full = np.vstack([base, pad])
    else:
        full = base[:seq_len, :]

    id_seqpos = [f"{_id}_{i}" for i in range(full.shape[0])]
    out = pd.DataFrame(full, columns=target_cols)
    out.insert(0, "id_seqpos", id_seqpos)
    rows.append(out)

submission_full = pd.concat(rows, axis=0, ignore_index=True)

submission = test_set[["id_seqpos"]].merge(submission_full, on="id_seqpos", how="left")
for col in target_cols:
    submission[col] = pd.to_numeric(submission[col], errors="coerce").astype(float)
submission.fillna(0.0, inplace=True)

assert submission.shape[0] == test_set.shape[0]
assert list(submission.columns) == list(test_set.columns)

submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", submission.shape)
submission.head(10)

Wrote submission.csv with shape: (25680, 6)


,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.087121,0.180998,0.231937,0.131574,0.148763
1,id_00b436dec_1,0.087121,0.180998,0.231937,0.131574,0.148763
2,id_00b436dec_2,0.087121,0.180998,0.231937,0.131574,0.148763
3,id_00b436dec_3,0.087121,0.180998,0.231937,0.131574,0.148763
4,id_00b436dec_4,0.087121,0.180998,0.231937,0.131574,0.148763
5,id_00b436dec_5,0.087121,0.180998,0.231937,0.131574,0.148763
6,id_00b436dec_6,0.087121,0.180998,0.231937,0.131574,0.148763
7,id_00b436dec_7,0.087121,0.180998,0.231937,0.131574,0.148763
8,id_00b436dec_8,0.087121,0.180998,0.231937,0.131574,0.148763
9,id_00b436dec_9,0.087121,0.180998,0.231937,0.131574,0.148763
